# Configurable Tiny-p Simulation Study

This notebook is a configurable simulation harness for permutation-test tiny p-values.

It provides:
1. Scenario setups loaded from saved exact-scenario catalog (data + exact p-values)
2. Configurable `p0` handling (`true p` vs user guess), MCMC/SAMC/IID iteration budgets, burn-in, and seeds
3. Cross-method simulation + diagnostics/comparison
4. MCMC-IS beta diagnostics (replicated across beta values)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from perm_pval.core.problem import PermutationTestProblem
from perm_pval.methods.random_sampling import run_random_sampling
from perm_pval.methods.mcmc_is import run_mcmc_is
from perm_pval.methods.samc import run_samc
from perm_pval.methods.beta_tuning import (
    estimate_scale_T,
    iid_pilot_statistics,
    init_beta_from_iid_pilot,
    make_short_chain_q_runner,
    tune_beta_to_target_q,
)
from perm_pval.experiments.exact_scenarios import load_saved_exact_scenarios

print(f"Using project root: {project_root}")


## Configuration

Edit this cell only for most runs.


In [ ]:
# ---------- Global run mode ----------
FAST_MODE = False

# ---------- Scenario selection ----------
# These keys come from results/exact_scenarios/v1/catalog.json.
# Canonical keys currently include:
# "hypergeom_1e5", "hypergeom_1e7", "rank_sum_dp_n40", "linear_stat_dp_n40",
# "poisson_diffmeans_righttail_tiny_n200", "bruteforce_welch_nonextreme_n22"
SCENARIO_KEYS_TO_RUN = [
    "bruteforce_welch_nonextreme_n22",
    "hypergeom_1e7",
    "rank_sum_dp_n40",
    "linear_stat_dp_n40",
    "poisson_diffmeans_righttail_tiny_n200",
]
MIN_TAIL_STATES = 2  # Guardrail: every selected scenario must have at least this many tail states.
EXACT_SCENARIO_CATALOG = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"

# ---------- Reproducibility ----------
BASE_SEED = 12345

# ---------- p0/q_target control ----------
USE_TRUE_P0_FOR_Q_TARGET = True
P0_GUESS = 1e-8
D_ALPHA = 0.25

# ---------- Repeats ----------
N_REPEATS = 5
BETA_SWEEP_REPEATS = 5

# ---------- Per-method simulation budgets ----------
# IID
IID_SAMPLES = 40_000 if FAST_MODE else 1_000_000

# MCMC-IS production
MCMC_CHAINS = 2
MCMC_STEPS_PER_CHAIN = 20_000 if FAST_MODE else 500_000
MCMC_BURN_IN = int(0.20 * MCMC_STEPS_PER_CHAIN)
MCMC_THIN = 1
MCMC_ESTIMATE_VARIANCE = True
MCMC_OBM_BATCH_SIZE = None
MCMC_TILT_MODE = "smooth_hinge"  # "smooth_hinge" or "step"
MCMC_PROPOSAL_FRAC = 0.075
MCMC_PROPOSAL_SWAPS = None  # optional override

# MCMC-IS beta initialization/tuning
MCMC_PILOT_SAMPLES = 4_000 if FAST_MODE else 20_000
MCMC_SCALE_METHOD = "sd"  # "sd" or "mad"
MCMC_BETA_MAX_INIT = 1e6
MCMC_TUNE_STEPS = 3_000 if FAST_MODE else 12_000
MCMC_TUNE_BURN_IN = int(0.20 * MCMC_TUNE_STEPS)
MCMC_TUNE_THIN = 2
MCMC_TUNE_BRACKET_FACTOR = 2.0
MCMC_TUNE_TOL_REL = 0.20
MCMC_TUNE_MAX_BRACKET = 12
MCMC_TUNE_MAX_BISECT = 12
MCMC_TUNE_REPLICATE = 1
MCMC_TUNE_REUSE_STATE = True
MCMC_TUNING_BUDGET_MODE = "per_run"  # "per_run" or "amortized"
BETA_OVERRIDE = None  # set float to force beta

# Shared beta multipliers for local scan + beta diagnostics
BETA_MULTIPLIERS = [0.70, 0.90, 1.00, 1.15, 1.35]

# MCMC-IS local beta scan (cheap)
BETA_LOCAL_SCAN_ENABLED = True
BETA_LOCAL_SCAN_MULTIPLIERS = list(BETA_MULTIPLIERS)
BETA_LOCAL_SCAN_TOTAL_STEPS = 20_000 if FAST_MODE else 80_000
BETA_LOCAL_SCAN_CHAINS = 2
BETA_LOCAL_SCAN_BURN_IN_FRAC = 0.20
BETA_LOCAL_SCAN_THIN = 1
# Score rule: variance-only (no ESS/q penalty).

# SAMC production
SAMC_STEPS = 40_000 if FAST_MODE else 1_000_000
SAMC_BURN_IN = int(0.20 * SAMC_STEPS)
SAMC_N_BINS = 40
SAMC_T0 = 1_000.0
SAMC_TRACE_EVERY = 200
SAMC_PROPOSAL_FRAC = MCMC_PROPOSAL_FRAC
SAMC_PROPOSAL_SWAPS = MCMC_PROPOSAL_SWAPS
SAMC_CONVERGENCE_TOL = 20.0
SAMC_LAMBDA_MIN_PILOT = 2_000 if FAST_MODE else 10_000

# ---------- Beta sweep diagnostics ----------
# Uses the same multiplier ladder as local beta scan.
BETA_SWEEP_MULTIPLIERS = list(BETA_MULTIPLIERS)
BETA_SWEEP_TOTAL_STEPS = 30_000 if FAST_MODE else 1_000_000
BETA_SWEEP_CHAINS = 2
BETA_SWEEP_BURN_IN_FRAC = 0.20
BETA_SWEEP_THIN = 1

# ---------- IID statistic density diagnostics ----------
IID_DENSITY_SAMPLES = 20_000 if FAST_MODE else 120_000

# ---------- Save policy ----------
SAVE_OUTPUTS = True
OUTPUT_DIR = project_root / "results" / "simulation_study_configurable"

# Smoke policy: smoke runs are never saved.
SMOKE_RUN = FAST_MODE
AUTO_DETECT_SMOKE = True
if AUTO_DETECT_SMOKE:
    looks_like_smoke = (
        N_REPEATS <= 1
        and BETA_SWEEP_REPEATS <= 1
        and IID_SAMPLES <= 10_000
        and MCMC_STEPS_PER_CHAIN <= 10_000
        and SAMC_STEPS <= 10_000
        and BETA_SWEEP_TOTAL_STEPS <= 10_000
    )
    SMOKE_RUN = bool(SMOKE_RUN or looks_like_smoke)

if SMOKE_RUN and SAVE_OUTPUTS:
    print("SMOKE_RUN=True -> forcing SAVE_OUTPUTS=False")
    SAVE_OUTPUTS = True

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_KEYS_TO_RUN": SCENARIO_KEYS_TO_RUN,
    "MIN_TAIL_STATES": MIN_TAIL_STATES,
    "EXACT_SCENARIO_CATALOG": str(EXACT_SCENARIO_CATALOG),
    "USE_TRUE_P0_FOR_Q_TARGET": USE_TRUE_P0_FOR_Q_TARGET,
    "P0_GUESS": P0_GUESS,
    "D_ALPHA": D_ALPHA,
    "N_REPEATS": N_REPEATS,
    "BETA_SWEEP_REPEATS": BETA_SWEEP_REPEATS,
    "IID_SAMPLES": IID_SAMPLES,
    "MCMC_CHAINS": MCMC_CHAINS,
    "MCMC_STEPS_PER_CHAIN": MCMC_STEPS_PER_CHAIN,
    "MCMC_BURN_IN": MCMC_BURN_IN,
    "MCMC_TILT_MODE": MCMC_TILT_MODE,
    "MCMC_PROPOSAL_FRAC": MCMC_PROPOSAL_FRAC,
    "MCMC_PROPOSAL_SWAPS": MCMC_PROPOSAL_SWAPS,
    "MCMC_TUNING_BUDGET_MODE": MCMC_TUNING_BUDGET_MODE,
    "BETA_LOCAL_SCAN_ENABLED": BETA_LOCAL_SCAN_ENABLED,
    "BETA_MULTIPLIERS": BETA_MULTIPLIERS,
    "BETA_LOCAL_SCAN_MULTIPLIERS": BETA_LOCAL_SCAN_MULTIPLIERS,
    "SAMC_STEPS": SAMC_STEPS,
    "SAMC_BURN_IN": SAMC_BURN_IN,
    "SAMC_PROPOSAL_FRAC": SAMC_PROPOSAL_FRAC,
    "SAMC_PROPOSAL_SWAPS": SAMC_PROPOSAL_SWAPS,
    "BETA_SWEEP_MULTIPLIERS": BETA_SWEEP_MULTIPLIERS,
    "SMOKE_RUN": SMOKE_RUN,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
}, indent=2))


## Scenario Library (Observations + Exact p-value)


In [ ]:
@dataclass
class Scenario:
    key: str
    display_name: str
    problem: PermutationTestProblem
    exact_p: float
    exact_tail_hits: int
    exact_n_perm: int
    exact_method: str


# Backward-compatible aliases for older notebook keys.
SCENARIO_KEY_ALIASES = {
    "small_bruteforce": "bruteforce_welch_nonextreme_n22",
    "binary_hypergeom_1e5": "hypergeom_1e5",
    "binary_hypergeom": "hypergeom_1e7",
    "rank_linear_x": "rank_sum_dp_n40",
    "linear_nonlinear_x": "linear_stat_dp_n40",
}


def canonical_scenario_key(key: str) -> str:
    return SCENARIO_KEY_ALIASES.get(key, key)


def load_scenarios_from_catalog(catalog_path: Path) -> dict[str, Scenario]:
    saved = load_saved_exact_scenarios(catalog_path)
    out: dict[str, Scenario] = {}
    for s in saved:
        out[s.key] = Scenario(
            key=s.key,
            display_name=s.description,
            problem=s.problem,
            exact_p=float(s.exact_p_value),
            exact_tail_hits=int(s.tail_hits),
            exact_n_perm=int(s.n_permutations),
            exact_method=s.exact_method,
        )
    return out


if not EXACT_SCENARIO_CATALOG.exists():
    raise FileNotFoundError(
        f"Exact scenario catalog not found: {EXACT_SCENARIO_CATALOG}. "
        "Generate it with: python -m perm_pval.experiments.generate_exact_scenarios"
    )

all_scenarios = load_scenarios_from_catalog(EXACT_SCENARIO_CATALOG)
requested_keys = [canonical_scenario_key(k) for k in SCENARIO_KEYS_TO_RUN]
missing = [k for k in requested_keys if k not in all_scenarios]
if missing:
    raise KeyError(
        f"Unknown scenario keys: {missing}. Available keys: {sorted(all_scenarios.keys())}"
    )

selected_scenarios = [all_scenarios[k] for k in requested_keys]
for sc in selected_scenarios:
    if sc.exact_tail_hits < MIN_TAIL_STATES:
        raise ValueError(
            f"Scenario '{sc.key}' has only {sc.exact_tail_hits} tail state(s). "
            f"Increase tail multiplicity or lower MIN_TAIL_STATES (currently {MIN_TAIL_STATES})."
        )

print(f"Loaded exact scenarios from: {EXACT_SCENARIO_CATALOG}")
print("Selected scenarios:")
for sc in selected_scenarios:
    print(json.dumps({
        "key": sc.key,
        "display_name": sc.display_name,
        "exact_p": sc.exact_p,
        "tail_hits": sc.exact_tail_hits,
        "n_perm": sc.exact_n_perm,
        "exact_method": sc.exact_method,
    }, indent=2))


## Helpers: Tuning, Methods, Summaries


In [ ]:
def samc_variance_proxy(p_hat: float, n_steps: int, burn_in: int) -> float:
    n_eff = max(int(n_steps - burn_in), 1)
    return float(max(p_hat * (1.0 - p_hat) / n_eff, 0.0))


def _count_short_chain_calls_from_history(history: list[dict]) -> int:
    n = 0
    for rec in history:
        stage = str(rec.get("stage", ""))
        if stage == "init" or stage.startswith("bracket_") or ":rep" in stage:
            n += 1
    return n


def _mcmc_eval_count(n_steps: int, n_chains: int) -> int:
    return int(n_chains * (n_steps + 1))


def _budget_note_line() -> str:
    mcmc_prod = int(MCMC_CHAINS * MCMC_STEPS_PER_CHAIN)
    return (
        f"Production iterations: IID={int(IID_SAMPLES):,}, MCMC-IS={mcmc_prod:,}, SAMC={int(SAMC_STEPS):,}; repeats={int(N_REPEATS)}. "
        "MCMC-IS setup (init+tuning+local scan) is counted in eval/time diagnostics."
    )


def _positive_for_plot(values: np.ndarray) -> np.ndarray:
    v = np.asarray(values, dtype=float).copy()
    v[~np.isfinite(v)] = np.nan
    v[v <= 0.0] = np.nan
    if np.all(np.isnan(v)):
        return np.array([np.nan], dtype=float)
    return v


def _set_log_ylim(ax, arrays: list[np.ndarray], q_lo: float = 0.05, q_hi: float = 0.95, pad: float = 1.35) -> None:
    vals = []
    for a in arrays:
        arr = np.asarray(a, dtype=float)
        arr = arr[np.isfinite(arr) & (arr > 0.0)]
        if arr.size:
            vals.append(arr)
    if not vals:
        return
    flat = np.concatenate(vals)
    lo = float(np.quantile(flat, q_lo))
    hi = float(np.quantile(flat, q_hi))
    if not np.isfinite(lo) or not np.isfinite(hi):
        return
    if hi <= lo:
        hi = lo * 1.10
    lo = max(lo / pad, np.min(flat) / pad)
    hi = hi * pad
    if lo > 0.0 and hi > lo:
        ax.set_ylim(lo, hi)


def _set_linear_ylim(ax, arrays: list[np.ndarray], q_lo: float = 0.02, q_hi: float = 0.98, pad: float = 0.12) -> None:
    vals = []
    for a in arrays:
        arr = np.asarray(a, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size:
            vals.append(arr)
    if not vals:
        return
    flat = np.concatenate(vals)
    lo = float(np.quantile(flat, q_lo))
    hi = float(np.quantile(flat, q_hi))
    if hi <= lo:
        eps = max(abs(lo), 1.0) * 0.05
        lo, hi = lo - eps, hi + eps
    span = hi - lo
    ax.set_ylim(lo - pad * span, hi + pad * span)


def _stat_label(problem: PermutationTestProblem) -> str:
    fn = getattr(problem.statistic, "__name__", "statistic")
    return str(fn)


def local_beta_scan(
    problem: PermutationTestProblem,
    *,
    beta_center: float,
    sigma_t: float,
    seed: int,
) -> dict:
    if not BETA_LOCAL_SCAN_ENABLED:
        return {
            "enabled": False,
            "selected_beta": float(beta_center),
            "rows": [],
            "scan_eval_total": 0,
            "scan_wall_time_sec": 0.0,
        }

    if beta_center <= 0.0:
        return {
            "enabled": True,
            "selected_beta": float(beta_center),
            "rows": [],
            "scan_eval_total": 0,
            "scan_wall_time_sec": 0.0,
            "note": "beta_center<=0; scan skipped",
        }

    betas = [float(beta_center * m) for m in BETA_LOCAL_SCAN_MULTIPLIERS if float(beta_center * m) > 0.0]
    if not betas:
        return {
            "enabled": True,
            "selected_beta": float(beta_center),
            "rows": [],
            "scan_eval_total": 0,
            "scan_wall_time_sec": 0.0,
            "note": "no positive beta candidates",
        }

    steps_per_chain = max(int(BETA_LOCAL_SCAN_TOTAL_STEPS // BETA_LOCAL_SCAN_CHAINS), 1)
    burn_in = min(int(BETA_LOCAL_SCAN_BURN_IN_FRAC * steps_per_chain), max(steps_per_chain - 1, 0))

    rows = []
    t0 = time.perf_counter()
    for j, beta in enumerate(betas):
        res = run_mcmc_is(
            problem,
            beta=beta,
            sigma_t=sigma_t,
            n_steps=steps_per_chain,
            burn_in=burn_in,
            thin=BETA_LOCAL_SCAN_THIN,
            n_chains=BETA_LOCAL_SCAN_CHAINS,
            seed=seed + 500 * j,
            init="random",
            tilt_mode=MCMC_TILT_MODE,
            proposal_size=(MCMC_PROPOSAL_SWAPS if MCMC_PROPOSAL_SWAPS is not None else MCMC_PROPOSAL_FRAC),
            estimate_variance=True,
            obm_batch_size=MCMC_OBM_BATCH_SIZE,
        )
        q_hat = float(res.tail_share_raw_sample)
        ess = float(res.ess)
        var_hat = float(res.snis_variance_obm) if (res.snis_variance_obm is not None and np.isfinite(res.snis_variance_obm) and res.snis_variance_obm > 0.0) else np.nan
        weight_cv = float(res.weight_summary.cv)

        # Variance-only scoring: choose beta with minimum estimated SNIS variance.
        if np.isfinite(var_hat) and var_hat > 0.0:
            score = float(np.log(var_hat))
        else:
            score = float("inf")

        rows.append({
            "beta": float(beta),
            "estimate": float(res.estimate),
            "variance_estimate": var_hat,
            "q_hat": q_hat,
            "ess": ess,
            "acceptance_rate": float(res.overall_acceptance_rate),
            "weight_cv": weight_cv,
            "score": float(score),
        })

    scan_wall_time_sec = float(time.perf_counter() - t0)

    # Pick minimum finite score; fallback to center.
    finite_rows = [r for r in rows if np.isfinite(r["score"])]
    if finite_rows:
        best = min(finite_rows, key=lambda r: r["score"])
        selected_beta = float(best["beta"])
        selected_reason = "min_scan_variance"
    else:
        selected_beta = float(beta_center)
        selected_reason = "fallback_beta_center"

    scan_eval_total = int(len(betas) * _mcmc_eval_count(steps_per_chain, BETA_LOCAL_SCAN_CHAINS))

    return {
        "enabled": True,
        "selected_beta": selected_beta,
        "selected_reason": selected_reason,
        "steps_per_chain": int(steps_per_chain),
        "burn_in": int(burn_in),
        "scan_eval_total": int(scan_eval_total),
        "scan_wall_time_sec": scan_wall_time_sec,
        "rows": rows,
    }


def build_beta_workflow(problem: PermutationTestProblem, exact_p: float, *, seed: int) -> dict:
    p0_for_qtarget = float(exact_p) if USE_TRUE_P0_FOR_Q_TARGET else float(P0_GUESS)
    q_target = float(p0_for_qtarget ** D_ALPHA)

    t0 = time.perf_counter()
    pilot_t = iid_pilot_statistics(problem, n_samples=MCMC_PILOT_SAMPLES, seed=seed)
    sigma_t = estimate_scale_T(pilot_t, method=MCMC_SCALE_METHOD)

    beta0_formula = float(np.sqrt(np.log(1.0 / exact_p)))
    beta0_laplace = init_beta_from_iid_pilot(
        pilot_T=pilot_t,
        T_obs=problem.t_obs,
        sigma_T=sigma_t,
        p0=p0_for_qtarget,
        q_target=q_target,
        beta_max=MCMC_BETA_MAX_INIT,
    )

    runner = make_short_chain_q_runner(
        problem,
        sigma_T=sigma_t,
        thin=MCMC_TUNE_THIN,
        proposal_size=(MCMC_PROPOSAL_SWAPS if MCMC_PROPOSAL_SWAPS is not None else MCMC_PROPOSAL_FRAC),
        seed=seed + 1,
    )
    tuning = tune_beta_to_target_q(
        run_short_chain_fn=runner,
        init_state=problem.y_obs,
        beta0=beta0_laplace,
        q_target=q_target,
        n_steps=MCMC_TUNE_STEPS,
        burn_in=MCMC_TUNE_BURN_IN,
        bracket_factor=MCMC_TUNE_BRACKET_FACTOR,
        tol_rel=MCMC_TUNE_TOL_REL,
        max_bracket_iter=MCMC_TUNE_MAX_BRACKET,
        max_bisect_iter=MCMC_TUNE_MAX_BISECT,
        replicate=MCMC_TUNE_REPLICATE,
        reuse_state=MCMC_TUNE_REUSE_STATE,
    )

    tuning_wall_time_sec = float(time.perf_counter() - t0)
    n_short_calls = _count_short_chain_calls_from_history(tuning["history"])
    tuning_eval_total = int(MCMC_PILOT_SAMPLES + n_short_calls * (MCMC_TUNE_STEPS + 1))

    beta_tuned = float(tuning["beta_hat"])

    if BETA_OVERRIDE is not None:
        beta_used = float(BETA_OVERRIDE)
        scan = {
            "enabled": False,
            "selected_beta": beta_used,
            "rows": [],
            "scan_eval_total": 0,
            "scan_wall_time_sec": 0.0,
            "selected_reason": "beta_override",
        }
    else:
        scan = local_beta_scan(
            problem,
            beta_center=beta_tuned,
            sigma_t=float(sigma_t),
            seed=seed + 7,
        )
        beta_used = float(scan["selected_beta"])

    return {
        "beta0_formula": beta0_formula,
        "beta0_laplace": float(beta0_laplace),
        "beta_hat_tuned": beta_tuned,
        "beta_used": beta_used,
        "sigma_t": float(sigma_t),
        "p0_for_qtarget": p0_for_qtarget,
        "q_target": q_target,
        "q_hat_beta_hat": float(tuning["q_hat"]),
        "bracket_succeeded": bool(tuning["bracket_succeeded"]),
        "n_short_chain_calls": int(n_short_calls),
        "tuning_eval_total": int(tuning_eval_total),
        "tuning_wall_time_sec": tuning_wall_time_sec,
        "local_scan": scan,
        "history_tail": tuning["history"][-5:],
    }


def tune_samc_setup(problem: PermutationTestProblem, *, seed: int) -> dict:
    pilot_t = iid_pilot_statistics(problem, n_samples=SAMC_LAMBDA_MIN_PILOT, seed=seed)
    lam_min = float(np.min(pilot_t))
    if lam_min >= problem.t_obs:
        lam_min = float(problem.t_obs - 1.0)
    finite_edges = np.linspace(lam_min, float(problem.t_obs), SAMC_N_BINS, dtype=float)
    bin_edges = np.concatenate([finite_edges, np.asarray([np.inf], dtype=float)])
    return {"lambda_min": lam_min, "bin_edges": bin_edges}


def plot_iid_stat_density(problem: PermutationTestProblem, scenario_name: str, *, exact_p: float, n_samples: int, seed: int, save_path: Path | None = None) -> dict:
    t_vals = iid_pilot_statistics(problem, n_samples=n_samples, seed=seed)
    t_obs = float(problem.t_obs)

    fig, ax = plt.subplots(1, 1, figsize=(8.4, 4.8))
    hist_vals, bin_edges, _ = ax.hist(
        t_vals,
        bins=70,
        density=True,
        alpha=0.35,
        color="#4e79a7",
        edgecolor="none",
    )
    ax.axvline(t_obs, color="black", linestyle="--", linewidth=1.3, label=f"T_obs={t_obs:.4g}")

    # Shade tail bins using the exact tail predicate to avoid orientation mistakes.
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    tail_mask = np.asarray([problem.is_in_tail(float(c)) for c in centers], dtype=bool)
    if np.any(tail_mask):
        widths = np.diff(bin_edges)
        ax.bar(
            bin_edges[:-1][tail_mask],
            hist_vals[tail_mask],
            width=widths[tail_mask],
            align="edge",
            color="#f28e2b",
            alpha=0.18,
            edgecolor="none",
            label="tail region",
            zorder=3,
        )

    ax.set_title(f"IID statistic density: {scenario_name}")
    ax.set_xlabel(_stat_label(problem))
    ax.set_ylabel("density")
    ax.legend(loc="best")

    fig.suptitle(f"Empirical null statistic shape (n={n_samples:,}, true p={exact_p:.3e})")
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.show()

    return {
        "n_samples": int(n_samples),
        "t_obs": t_obs,
        "t_min": float(np.nanmin(t_vals)),
        "t_max": float(np.nanmax(t_vals)),
        "t_mean": float(np.nanmean(t_vals)),
        "t_sd": float(np.nanstd(t_vals, ddof=1)),
    }



def run_one_rep(problem: PermutationTestProblem, exact_p: float, *, rep_seed: int, beta_workflow: dict, samc_setup: dict, n_repeats_for_amort: int) -> list[dict]:
    rows = []

    if MCMC_TUNING_BUDGET_MODE not in {"per_run", "amortized"}:
        raise ValueError("MCMC_TUNING_BUDGET_MODE must be 'per_run' or 'amortized'.")

    tune_eval = float(beta_workflow.get("tuning_eval_total", 0.0))
    tune_time = float(beta_workflow.get("tuning_wall_time_sec", 0.0))
    scan_eval = float(beta_workflow.get("local_scan", {}).get("scan_eval_total", 0.0))
    scan_time = float(beta_workflow.get("local_scan", {}).get("scan_wall_time_sec", 0.0))

    if MCMC_TUNING_BUDGET_MODE == "per_run":
        setup_eval_charge = tune_eval + scan_eval
        setup_time_charge = tune_time + scan_time
    else:
        denom = max(int(n_repeats_for_amort), 1)
        setup_eval_charge = (tune_eval + scan_eval) / denom
        setup_time_charge = (tune_time + scan_time) / denom

    # IID
    t0 = time.perf_counter()
    iid = run_random_sampling(problem, n_samples=IID_SAMPLES, seed=rep_seed)
    iid_dt = float(time.perf_counter() - t0)
    rows.append({
        "method": "iid",
        "estimate": float(iid.estimate),
        "variance_estimate": float(iid.standard_error ** 2),
        "tail_hits": int(iid.tail_hits),
        "tail_share_raw": float(iid.estimate),
        "zero_hits": int(iid.tail_hits == 0),
        "wall_time_sec": iid_dt,
        "wall_time_incl_tuning_sec": iid_dt,
        "eval_excl_tuning": float(iid.n_samples),
        "eval_incl_tuning": float(iid.n_samples),
    })

    # MCMC-IS
    t0 = time.perf_counter()
    mcmc = run_mcmc_is(
        problem,
        beta=beta_workflow["beta_used"],
        sigma_t=beta_workflow["sigma_t"],
        n_steps=MCMC_STEPS_PER_CHAIN,
        burn_in=MCMC_BURN_IN,
        thin=MCMC_THIN,
        n_chains=MCMC_CHAINS,
        seed=rep_seed + 1,
        init="random",
        tilt_mode=MCMC_TILT_MODE,
        proposal_size=(MCMC_PROPOSAL_SWAPS if MCMC_PROPOSAL_SWAPS is not None else MCMC_PROPOSAL_FRAC),
        estimate_variance=MCMC_ESTIMATE_VARIANCE,
        obm_batch_size=MCMC_OBM_BATCH_SIZE,
    )
    mcmc_dt = float(time.perf_counter() - t0)
    mcmc_eval_excl = float(_mcmc_eval_count(MCMC_STEPS_PER_CHAIN, MCMC_CHAINS))
    rows.append({
        "method": "mcmc_is",
        "estimate": float(mcmc.estimate),
        "variance_estimate": float(mcmc.snis_variance_obm) if (mcmc.snis_variance_obm is not None and np.isfinite(mcmc.snis_variance_obm)) else np.nan,
        "snis_mcse_obm": float(mcmc.snis_mcse_obm) if (mcmc.snis_mcse_obm is not None and np.isfinite(mcmc.snis_mcse_obm)) else np.nan,
        "tail_hits": int(mcmc.tail_hits_weighted_sample),
        "tail_share_raw": float(mcmc.tail_share_raw_sample),
        "ess": float(mcmc.ess),
        "acceptance_rate": float(mcmc.overall_acceptance_rate),
        "weight_cv": float(mcmc.weight_summary.cv),
        "beta": float(mcmc.beta),
        "sigma_t": float(mcmc.sigma_t),
        "tilt_mode": str(mcmc.tilt_mode),
        "wall_time_sec": mcmc_dt,
        "wall_time_incl_tuning_sec": mcmc_dt + setup_time_charge,
        "eval_excl_tuning": mcmc_eval_excl,
        "eval_incl_tuning": mcmc_eval_excl + setup_eval_charge,
    })

    # SAMC
    t0 = time.perf_counter()
    samc = run_samc(
        problem,
        n_steps=SAMC_STEPS,
        burn_in=SAMC_BURN_IN,
        bin_edges=samc_setup["bin_edges"],
        seed=rep_seed + 2,
        init="random",
        t0=SAMC_T0,
        trace_every=SAMC_TRACE_EVERY,
        proposal_size=(SAMC_PROPOSAL_SWAPS if SAMC_PROPOSAL_SWAPS is not None else SAMC_PROPOSAL_FRAC),
        convergence_tolerance=SAMC_CONVERGENCE_TOL,
    )
    samc_dt = float(time.perf_counter() - t0)
    samc_eval = float(SAMC_STEPS + 1)
    rows.append({
        "method": "samc",
        "estimate": float(samc.estimate),
        "variance_estimate": samc_variance_proxy(float(samc.estimate), SAMC_STEPS, SAMC_BURN_IN),
        "acceptance_rate": float(samc.acceptance_rate),
        "samc_max_rel_freq_error": float(samc.max_abs_relative_frequency_error),
        "samc_converged": int(samc.convergence_reached),
        "samc_pi0": float(samc.pi0_adjustment),
        "samc_empty_bins": int(samc.empty_bin_indices.size),
        "wall_time_sec": samc_dt,
        "wall_time_incl_tuning_sec": samc_dt,
        "eval_excl_tuning": samc_eval,
        "eval_incl_tuning": samc_eval,
    })

    for row in rows:
        est = float(row["estimate"])
        row["exact_p"] = float(exact_p)
        row["bias"] = float(est - exact_p)
        row["squared_error"] = float((est - exact_p) ** 2)
        row["root_squared_error"] = float(np.sqrt((est - exact_p) ** 2))
        row["rel_error"] = float((est - exact_p) / exact_p)
        if est > 0.0:
            row["abs_log10_error"] = float(abs(np.log10(est) - np.log10(exact_p)))
        else:
            row["abs_log10_error"] = np.nan

    return rows


def summarize_records(records: list[dict]) -> list[dict]:
    out = []
    methods = sorted({r["method"] for r in records})
    for m in methods:
        sub = [r for r in records if r["method"] == m]
        est = np.asarray([r["estimate"] for r in sub], dtype=float)
        sq = np.asarray([r["squared_error"] for r in sub], dtype=float)
        var = np.asarray([r["variance_estimate"] for r in sub], dtype=float)
        exact_p = float(sub[0]["exact_p"])

        emp_var = float(np.var(est, ddof=1)) if est.size > 1 else np.nan
        var_finite = var[np.isfinite(var) & (var > 0.0)]
        mean_var_hat = float(np.mean(var_finite)) if var_finite.size else np.nan
        var_calib = float(emp_var / mean_var_hat) if (np.isfinite(emp_var) and np.isfinite(mean_var_hat) and mean_var_hat > 0.0) else np.nan

        abs_log = np.asarray([r.get("abs_log10_error", np.nan) for r in sub], dtype=float)
        abs_log = abs_log[np.isfinite(abs_log)]

        out.append({
            "method": m,
            "n_runs": int(est.size),
            "exact_p": exact_p,
            "mean_estimate": float(np.mean(est)),
            "median_estimate": float(np.median(est)),
            "bias": float(np.mean(est) - exact_p),
            "rel_bias": float((np.mean(est) - exact_p) / exact_p),
            "rmse": float(np.sqrt(np.mean(sq))),
            "mean_abs_log10_error": float(np.mean(abs_log)) if abs_log.size else np.nan,
            "empirical_var": emp_var,
            "mean_variance_estimate": mean_var_hat,
            "var_calibration_ratio": var_calib,
            "mean_wall_time_sec": float(np.mean([r["wall_time_sec"] for r in sub])),
            "mean_wall_time_incl_tuning_sec": float(np.mean([r["wall_time_incl_tuning_sec"] for r in sub])),
            "mean_eval_excl_tuning": float(np.mean([r["eval_excl_tuning"] for r in sub])),
            "mean_eval_incl_tuning": float(np.mean([r["eval_incl_tuning"] for r in sub])),
            "mean_q_tilt_tail_share": float(np.mean([r.get("tail_share_raw", np.nan) for r in sub if np.isfinite(r.get("tail_share_raw", np.nan))])) if any(np.isfinite(r.get("tail_share_raw", np.nan)) for r in sub) else np.nan,
            "mean_ess": float(np.mean([r.get("ess", np.nan) for r in sub if np.isfinite(r.get("ess", np.nan))])) if any(np.isfinite(r.get("ess", np.nan)) for r in sub) else np.nan,
            "mean_acceptance_rate": float(np.mean([r.get("acceptance_rate", np.nan) for r in sub if np.isfinite(r.get("acceptance_rate", np.nan))])) if any(np.isfinite(r.get("acceptance_rate", np.nan)) for r in sub) else np.nan,
            "mean_zero_rate": float(np.mean([r.get("zero_hits", np.nan) for r in sub if np.isfinite(r.get("zero_hits", np.nan))])) if any(np.isfinite(r.get("zero_hits", np.nan)) for r in sub) else np.nan,
            "mean_weight_cv": float(np.mean([r.get("weight_cv", np.nan) for r in sub if np.isfinite(r.get("weight_cv", np.nan))])) if any(np.isfinite(r.get("weight_cv", np.nan)) for r in sub) else np.nan,
            "mean_samc_max_rel_freq_error": float(np.mean([r.get("samc_max_rel_freq_error", np.nan) for r in sub if np.isfinite(r.get("samc_max_rel_freq_error", np.nan))])) if any(np.isfinite(r.get("samc_max_rel_freq_error", np.nan)) for r in sub) else np.nan,
        })

    return out


def plot_cross_method(records: list[dict], summary: list[dict], scenario_name: str, exact_p: float, beta_workflow: dict | None = None, *, save_path: Path | None = None) -> None:
    methods = ["iid", "mcmc_is", "samc"]
    labels = ["IID", "MCMC-IS", "SAMC"]

    def _fmt(v: object, fmt: str) -> str:
        try:
            x = float(v)
            if not np.isfinite(x):
                return "nan"
            return format(x, fmt)
        except Exception:
            return "nan"

    beta_line = None
    if beta_workflow is not None:
        beta_line = (
            "MCMC-IS beta (laplace/tuned/used): "
            f"{_fmt(beta_workflow.get("beta0_laplace"), '.4g')} / "
            f"{_fmt(beta_workflow.get("beta_hat_tuned"), '.4g')} / "
            f"{_fmt(beta_workflow.get("beta_used"), '.4g')}; "
            f"q_target={_fmt(beta_workflow.get("q_target"), '.3e')}"
        )

    est_data = []
    var_data = []
    rmse_like_data = []

    for m in methods:
        sub = [r for r in records if r["method"] == m]
        est = np.asarray([r["estimate"] for r in sub], dtype=float)
        var_hat = np.asarray([r["variance_estimate"] for r in sub], dtype=float)
        rse = np.asarray([r["root_squared_error"] for r in sub], dtype=float)

        if m == "iid":
            tail_hits = np.asarray([r.get("tail_hits", 0) for r in sub], dtype=int)
            est[tail_hits <= 0] = np.nan
            var_hat[tail_hits <= 0] = np.nan
            rse[tail_hits <= 0] = np.nan

        est_data.append(_positive_for_plot(est))
        var_data.append(_positive_for_plot(var_hat))
        rmse_like_data.append(_positive_for_plot(rse))

    fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))

    axes[0].boxplot(est_data, tick_labels=labels, showfliers=False)
    axes[0].set_yscale("log")
    axes[0].axhline(exact_p, color="black", linestyle="--", linewidth=1.2, label=f"true p={exact_p:.2e}")
    _set_log_ylim(axes[0], est_data)
    axes[0].set_title("Estimator distribution")
    axes[0].set_ylabel("p_hat")
    axes[0].legend()

    axes[1].boxplot(var_data, tick_labels=labels, showfliers=False)
    axes[1].set_yscale("log")
    _set_log_ylim(axes[1], var_data)
    axes[1].set_title("Variance estimate distribution")
    axes[1].set_ylabel("var_hat")

    axes[2].boxplot(rmse_like_data, tick_labels=labels, showfliers=False)
    axes[2].set_yscale("log")
    _set_log_ylim(axes[2], rmse_like_data)
    axes[2].set_title("RMSE across repeats")
    axes[2].set_ylabel("root squared error")

    title = f"Cross-method comparison: {scenario_name} (true p={exact_p:.3e})\n{_budget_note_line()}"
    if beta_line is not None:
        title = f"{title}\n{beta_line}"
    fig.suptitle(title)
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.show()


## 1) Cross-Method Simulation + Diagnostics & Comparison


In [ ]:
# Cross-method simulation for all selected scenarios
if SAVE_OUTPUTS:
    run_tag = time.strftime("%Y%m%d_%H%M%S")
    run_dir = OUTPUT_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"Outputs will be written under: {run_dir}")
else:
    run_dir = None

cross_results: dict[str, dict] = {}

for s_idx, scenario in enumerate(selected_scenarios):
    print("=" * 80)
    print(f"Scenario: {scenario.key} | exact p={scenario.exact_p:.3e} | repeats={N_REPEATS}")

    beta_workflow = build_beta_workflow(scenario.problem, scenario.exact_p, seed=BASE_SEED + 100 + 10_000 * s_idx)
    samc_setup = tune_samc_setup(scenario.problem, seed=BASE_SEED + 200 + 10_000 * s_idx)

    print("Beta workflow:")
    print(json.dumps(beta_workflow, indent=2))

    run_records: list[dict] = []
    t0 = time.perf_counter()
    for r in range(N_REPEATS):
        rep_seed = BASE_SEED + 1000 * r + 50_000 * s_idx
        rows = run_one_rep(
            scenario.problem,
            scenario.exact_p,
            rep_seed=rep_seed,
            beta_workflow=beta_workflow,
            samc_setup=samc_setup,
            n_repeats_for_amort=N_REPEATS,
        )
        for row in rows:
            run_records.append({
                "scenario": scenario.key,
                "scenario_display": scenario.display_name,
                "replicate": int(r),
                **row,
            })

    summary_records = summarize_records(run_records)

    dt = time.perf_counter() - t0
    print(f"Completed in {dt/60.0:.2f} min with {len(run_records)} method runs.")
    print(f"Tail states: {scenario.exact_tail_hits} / {scenario.exact_n_perm}")
    print("Summary:")
    for row in summary_records:
        print(row)

    cross_plot_path = None
    if SAVE_OUTPUTS and run_dir is not None:
        cross_plot_path = run_dir / scenario.key / "plots" / "cross_method_comparison.png"

    plot_cross_method(
        run_records,
        summary_records,
        scenario.display_name,
        scenario.exact_p,
        beta_workflow=beta_workflow,
        save_path=cross_plot_path,
    )

    density_plot_path = None
    if SAVE_OUTPUTS and run_dir is not None:
        density_plot_path = run_dir / scenario.key / "plots" / "iid_stat_density.png"

    density_summary = plot_iid_stat_density(
        scenario.problem,
        scenario.display_name,
        exact_p=scenario.exact_p,
        n_samples=IID_DENSITY_SAMPLES,
        seed=BASE_SEED + 900_000 + 10_000 * s_idx,
        save_path=density_plot_path,
    )

    cross_results[scenario.key] = {
        "scenario": {
            "key": scenario.key,
            "display_name": scenario.display_name,
            "exact_p": float(scenario.exact_p),
            "exact_tail_hits": int(scenario.exact_tail_hits),
            "exact_n_perm": int(scenario.exact_n_perm),
            "exact_method": scenario.exact_method,
        },
        "beta_workflow": beta_workflow,
        "samc_setup": {
            "lambda_min": float(samc_setup["lambda_min"]),
            "n_bins": int(len(samc_setup["bin_edges"]) - 1),
            "bin_edges": [float(v) if np.isfinite(v) else "inf" for v in samc_setup["bin_edges"]],
        },
        "iid_density_summary": density_summary,
        "run_records": run_records,
        "summary_records": summary_records,
        "cross_plot_path": str(cross_plot_path) if cross_plot_path is not None else None,
        "density_plot_path": str(density_plot_path) if density_plot_path is not None else None,
        "elapsed_sec": float(dt),
    }


## 2) MCMC-IS Beta Diagnostics (Replicated Across Betas)


In [ ]:
def run_beta_diagnostics_sweep(
    problem: PermutationTestProblem,
    exact_p: float,
    *,
    beta_center: float,
    sigma_t: float,
    seed_base: int,
    beta_multipliers: list[float],
    n_repeats: int,
    total_steps: int,
    n_chains: int,
    burn_in_frac: float,
    thin: int,
    estimate_variance: bool,
    obm_batch_size: int | None,
) -> dict:
    steps_per_chain = max(total_steps // n_chains, 1)
    burn_in = min(int(burn_in_frac * steps_per_chain), max(steps_per_chain - 1, 0))

    rows: list[dict] = []
    for i, mult in enumerate(beta_multipliers):
        beta = float(beta_center * float(mult))
        for r in range(n_repeats):
            seed = seed_base + 1000 * i + 10 * r
            res = run_mcmc_is(
                problem,
                beta=beta,
                sigma_t=sigma_t,
                n_steps=steps_per_chain,
                burn_in=burn_in,
                thin=thin,
                n_chains=n_chains,
                seed=seed,
                init="random",
                tilt_mode=MCMC_TILT_MODE,
                proposal_size=(MCMC_PROPOSAL_SWAPS if MCMC_PROPOSAL_SWAPS is not None else MCMC_PROPOSAL_FRAC),
                estimate_variance=estimate_variance,
                obm_batch_size=obm_batch_size,
            )
            est = float(res.estimate)
            var_hat = res.snis_variance_obm if (res.snis_variance_obm is not None and np.isfinite(res.snis_variance_obm)) else np.nan
            rows.append({
                "beta": beta,
                "beta_multiplier": float(mult),
                "repeat": int(r),
                "seed": int(seed),
                "estimate": est,
                "variance_estimate": float(var_hat) if np.isfinite(var_hat) else np.nan,
                "snis_mcse_obm": float(res.snis_mcse_obm) if (res.snis_mcse_obm is not None and np.isfinite(res.snis_mcse_obm)) else np.nan,
                "q_tilt_tail_share": float(res.tail_share_raw_sample),
                "ess": float(res.ess),
                "acceptance_rate": float(res.overall_acceptance_rate),
                "weight_cv": float(res.weight_summary.cv),
                "abs_log10_error": float(abs(np.log10(est) - np.log10(exact_p))) if est > 0.0 else np.nan,
            })

    summary = []
    for beta in sorted({r["beta"] for r in rows}):
        sub = [r for r in rows if r["beta"] == beta]
        est = np.asarray([r["estimate"] for r in sub], dtype=float)
        var = np.asarray([r["variance_estimate"] for r in sub], dtype=float)
        q = np.asarray([r["q_tilt_tail_share"] for r in sub], dtype=float)
        ess = np.asarray([r["ess"] for r in sub], dtype=float)
        acc = np.asarray([r["acceptance_rate"] for r in sub], dtype=float)
        wcv = np.asarray([r["weight_cv"] for r in sub], dtype=float)
        abs_log = np.asarray([r["abs_log10_error"] for r in sub], dtype=float)
        abs_log = abs_log[np.isfinite(abs_log)]

        var_finite = var[np.isfinite(var) & (var > 0.0)]
        mean_est = float(np.mean(est))
        emp_var = float(np.var(est, ddof=1)) if est.size > 1 else float("nan")
        mean_var_hat = float(np.mean(var_finite)) if var_finite.size else float("nan")
        var_calib = float(emp_var / mean_var_hat) if (np.isfinite(emp_var) and np.isfinite(mean_var_hat) and mean_var_hat > 0.0) else float("nan")

        summary.append({
            "beta": float(beta),
            "beta_multiplier": float(beta / beta_center),
            "n_runs": int(est.size),
            "mean_estimate": mean_est,
            "median_estimate": float(np.median(est)),
            "bias": float(mean_est - exact_p),
            "rel_bias": float((mean_est - exact_p) / exact_p),
            "rmse": float(np.sqrt(np.mean((est - exact_p) ** 2))),
            "mean_abs_log10_error": float(np.mean(abs_log)) if abs_log.size else np.nan,
            "mean_variance_estimate": mean_var_hat,
            "empirical_var": emp_var,
            "var_calibration_ratio": var_calib,
            "mean_q_tilt_tail_share": float(np.mean(q)),
            "mean_ess": float(np.mean(ess)),
            "mean_acceptance_rate": float(np.mean(acc)),
            "mean_weight_cv": float(np.mean(wcv)),
        })

    return {
        "rows": rows,
        "summary": summary,
        "settings": {
            "steps_per_chain": int(steps_per_chain),
            "n_chains": int(n_chains),
            "burn_in": int(burn_in),
            "thin": int(thin),
            "n_repeats": int(n_repeats),
            "beta_center": float(beta_center),
            "sigma_t": float(sigma_t),
            "beta_multipliers": [float(v) for v in beta_multipliers],
        },
    }


def plot_beta_sweep(beta_sweep: dict, exact_p: float, scenario_name: str, *, save_path: Path | None = None) -> None:
    rows = beta_sweep["rows"]
    betas = sorted({r["beta"] for r in rows})
    labels = [f"{b:.3g}" for b in betas]

    q_data = [np.asarray([r["q_tilt_tail_share"] for r in rows if r["beta"] == b], dtype=float) for b in betas]
    est_data = [_positive_for_plot(np.asarray([r["estimate"] for r in rows if r["beta"] == b], dtype=float)) for b in betas]
    ess_data = [_positive_for_plot(np.asarray([r["ess"] for r in rows if r["beta"] == b], dtype=float)) for b in betas]
    acc_data = [np.asarray([r["acceptance_rate"] for r in rows if r["beta"] == b], dtype=float) for b in betas]
    wcv_data = [np.asarray([r["weight_cv"] for r in rows if r["beta"] == b], dtype=float) for b in betas]
    var_data = [_positive_for_plot(np.asarray([r["variance_estimate"] for r in rows if r["beta"] == b], dtype=float)) for b in betas]

    fig, axes = plt.subplots(2, 3, figsize=(18, 8.5))

    # Requested orientation: beta on x-axis, q on y-axis.
    axes[0, 0].boxplot(q_data, tick_labels=labels, showfliers=False)
    _set_linear_ylim(axes[0, 0], q_data)
    axes[0, 0].set_title("Tilted-tail occupancy q")
    axes[0, 0].set_xlabel("beta")
    axes[0, 0].set_ylabel("q = P_pi_beta(T >= T_obs)")

    axes[0, 1].boxplot(est_data, tick_labels=labels, showfliers=False)
    axes[0, 1].set_yscale("log")
    axes[0, 1].axhline(exact_p, color="black", linestyle="--", linewidth=1.2, label=f"true p={exact_p:.2e}")
    _set_log_ylim(axes[0, 1], est_data)
    axes[0, 1].set_title("Estimator across beta")
    axes[0, 1].set_xlabel("beta")
    axes[0, 1].set_ylabel("p_hat")
    axes[0, 1].legend(loc="best")

    axes[0, 2].boxplot(var_data, tick_labels=labels, showfliers=False)
    axes[0, 2].set_yscale("log")
    _set_log_ylim(axes[0, 2], var_data)
    axes[0, 2].set_title("Estimated variance across beta")
    axes[0, 2].set_xlabel("beta")
    axes[0, 2].set_ylabel("var_hat")

    axes[1, 0].boxplot(ess_data, tick_labels=labels, showfliers=False)
    axes[1, 0].set_yscale("log")
    _set_log_ylim(axes[1, 0], ess_data)
    axes[1, 0].set_title("ESS across beta")
    axes[1, 0].set_xlabel("beta")
    axes[1, 0].set_ylabel("ESS")

    axes[1, 1].boxplot(acc_data, tick_labels=labels, showfliers=False)
    axes[1, 1].set_title("Acceptance across beta")
    axes[1, 1].set_xlabel("beta")

    axes[1, 2].boxplot(wcv_data, tick_labels=labels, showfliers=False)
    axes[1, 2].set_title("Weight CV across beta")
    axes[1, 2].set_xlabel("beta")

    fig.suptitle(f"{scenario_name}: MCMC-IS beta diagnostics (true p={exact_p:.3e}; repeats={BETA_SWEEP_REPEATS})")
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.show()


beta_results: dict[str, dict] = {}

for s_idx, scenario in enumerate(selected_scenarios):
    print("=" * 80)
    print(f"Beta diagnostics: {scenario.key}")
    beta_workflow = cross_results[scenario.key]["beta_workflow"]

    beta_sweep = run_beta_diagnostics_sweep(
        scenario.problem,
        scenario.exact_p,
        beta_center=float(beta_workflow["beta_used"]),
        sigma_t=float(beta_workflow["sigma_t"]),
        seed_base=BASE_SEED + 500_000 + 100_000 * s_idx,
        beta_multipliers=BETA_SWEEP_MULTIPLIERS,
        n_repeats=BETA_SWEEP_REPEATS,
        total_steps=BETA_SWEEP_TOTAL_STEPS,
        n_chains=BETA_SWEEP_CHAINS,
        burn_in_frac=BETA_SWEEP_BURN_IN_FRAC,
        thin=BETA_SWEEP_THIN,
        estimate_variance=MCMC_ESTIMATE_VARIANCE,
        obm_batch_size=MCMC_OBM_BATCH_SIZE,
    )

    beta_plot_path = None
    if SAVE_OUTPUTS and run_dir is not None:
        beta_plot_path = run_dir / scenario.key / "plots" / "beta_diagnostics.png"

    plot_beta_sweep(
        beta_sweep,
        scenario.exact_p,
        scenario.display_name,
        save_path=beta_plot_path,
    )

    print("Beta sweep summary:")
    for row in beta_sweep["summary"]:
        print(row)

    beta_results[scenario.key] = {
        "beta_sweep": beta_sweep,
        "beta_plot_path": str(beta_plot_path) if beta_plot_path is not None else None,
    }


## Optional Save: Run Records, Summary, Metadata


In [ ]:
if SAVE_OUTPUTS:
    if run_dir is None:
        run_tag = time.strftime("%Y%m%d_%H%M%S")
        run_dir = OUTPUT_DIR / run_tag
        run_dir.mkdir(parents=True, exist_ok=True)

    config_snapshot = {
        "FAST_MODE": FAST_MODE,
        "SCENARIO_KEYS_TO_RUN": SCENARIO_KEYS_TO_RUN,
        "EXACT_SCENARIO_CATALOG": str(EXACT_SCENARIO_CATALOG),
        "MIN_TAIL_STATES": MIN_TAIL_STATES,
        "USE_TRUE_P0_FOR_Q_TARGET": USE_TRUE_P0_FOR_Q_TARGET,
        "P0_GUESS": P0_GUESS,
        "D_ALPHA": D_ALPHA,
        "N_REPEATS": N_REPEATS,
        "BETA_SWEEP_REPEATS": BETA_SWEEP_REPEATS,
        "IID_SAMPLES": IID_SAMPLES,
        "MCMC_CHAINS": MCMC_CHAINS,
        "MCMC_STEPS_PER_CHAIN": MCMC_STEPS_PER_CHAIN,
        "MCMC_BURN_IN": MCMC_BURN_IN,
        "MCMC_THIN": MCMC_THIN,
        "MCMC_ESTIMATE_VARIANCE": MCMC_ESTIMATE_VARIANCE,
        "MCMC_TILT_MODE": MCMC_TILT_MODE,
        "MCMC_PROPOSAL_FRAC": MCMC_PROPOSAL_FRAC,
        "MCMC_PROPOSAL_SWAPS": MCMC_PROPOSAL_SWAPS,
        "MCMC_PILOT_SAMPLES": MCMC_PILOT_SAMPLES,
        "MCMC_SCALE_METHOD": MCMC_SCALE_METHOD,
        "MCMC_TUNE_STEPS": MCMC_TUNE_STEPS,
        "MCMC_TUNE_BURN_IN": MCMC_TUNE_BURN_IN,
        "MCMC_TUNE_THIN": MCMC_TUNE_THIN,
        "MCMC_TUNING_BUDGET_MODE": MCMC_TUNING_BUDGET_MODE,
        "BETA_LOCAL_SCAN_ENABLED": BETA_LOCAL_SCAN_ENABLED,
        "BETA_MULTIPLIERS": BETA_MULTIPLIERS,
        "BETA_LOCAL_SCAN_MULTIPLIERS": BETA_LOCAL_SCAN_MULTIPLIERS,
        "BETA_LOCAL_SCAN_TOTAL_STEPS": BETA_LOCAL_SCAN_TOTAL_STEPS,
        "SAMC_STEPS": SAMC_STEPS,
        "SAMC_BURN_IN": SAMC_BURN_IN,
        "SAMC_PROPOSAL_FRAC": SAMC_PROPOSAL_FRAC,
        "SAMC_PROPOSAL_SWAPS": SAMC_PROPOSAL_SWAPS,
        "SAMC_N_BINS": SAMC_N_BINS,
        "BETA_SWEEP_MULTIPLIERS": BETA_SWEEP_MULTIPLIERS,
        "BETA_SWEEP_TOTAL_STEPS": BETA_SWEEP_TOTAL_STEPS,
        "IID_DENSITY_SAMPLES": IID_DENSITY_SAMPLES,
        "SMOKE_RUN": SMOKE_RUN,
        "SAVE_OUTPUTS": SAVE_OUTPUTS,
    }
    (run_dir / "config.json").write_text(json.dumps(config_snapshot, indent=2), encoding="utf-8")

    for key, payload in cross_results.items():
        scenario_dir = run_dir / key
        scenario_dir.mkdir(parents=True, exist_ok=True)

        run_path = scenario_dir / "cross_run_records.jsonl"
        with run_path.open("w", encoding="utf-8") as f:
            for row in payload["run_records"]:
                f.write(json.dumps(row) + "\n")

        (scenario_dir / "cross_summary.json").write_text(
            json.dumps(payload["summary_records"], indent=2),
            encoding="utf-8",
        )

        scenario_meta = {
            "scenario": payload["scenario"],
            "beta_workflow": payload["beta_workflow"],
            "samc_setup": payload["samc_setup"],
            "iid_density_summary": payload.get("iid_density_summary"),
            "cross_plot_path": payload["cross_plot_path"],
            "density_plot_path": payload.get("density_plot_path"),
            "elapsed_sec_cross": payload["elapsed_sec"],
        }
        (scenario_dir / "cross_metadata.json").write_text(json.dumps(scenario_meta, indent=2), encoding="utf-8")

        beta_payload = beta_results.get(key)
        if beta_payload is not None:
            (scenario_dir / "beta_sweep.json").write_text(
                json.dumps(beta_payload["beta_sweep"], indent=2),
                encoding="utf-8",
            )
            (scenario_dir / "beta_metadata.json").write_text(
                json.dumps({"beta_plot_path": beta_payload["beta_plot_path"]}, indent=2),
                encoding="utf-8",
            )

    top_summary = {
        "scenarios": [payload["scenario"] for payload in cross_results.values()],
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "run_dir": str(run_dir),
    }
    (run_dir / "run_metadata.json").write_text(json.dumps(top_summary, indent=2), encoding="utf-8")
    print(f"Saved outputs to {run_dir}")
else:
    print("SAVE_OUTPUTS=False; nothing written to disk.")
